<a href="https://colab.research.google.com/github/areejtechcampus/AI-Agent/blob/main/Project_Barcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [139]:
!apt-get install -y libzbar0
!pip install pyzbar Pillow langgraph
!pip install -q -U langchain-openai


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libzbar0 is already the newest version (0.23.92-4build2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [140]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [141]:
import os
from pyzbar.pyzbar import decode
from PIL import Image
from typing import TypedDict, List
folder_path = '/content/drive/MyDrive/barcodes'
class AgentState(TypedDict):
   folder_path: str #
   extracted_urls: List[str]
   job_details: List[dict]
   final_report: str #




In [142]:
def extract_barcodes_node(state: AgentState):
    print("--- Exreracting links from barcodes---")
    urls = []
    folder = state['folder_path']

    for filename in os.listdir(folder):
        if filename.endswith((".png", ".jpg", ".jpeg")):
            path = os.path.join(folder, filename)

            decoded = decode(Image.open(path))
            for obj in decoded:
                url = obj.data.decode('utf-8')
                urls.append(url)


    return {"extracted_urls": list(set(urls))} #

In [143]:
import requests
from bs4 import BeautifulSoup



def scrape_jobs_node(state: AgentState):
    print(f"--- جاري تحليل {len(state['extracted_urls'])} روابط ---")
    jobs_info = []

    for url in state['extracted_urls']:
        try:

            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            text = soup.get_text(separator=' ', strip=True)[:2000]
            jobs_info.append({"url": url, "content": text})
        except Exception as e:

            print(f"فشل الوصول للرابط: {url} بسبب: {e}")

    return {"job_details": jobs_info}

In [144]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

def match_evaluator_node(state: AgentState):
    print("--- جاري تقييم الوظائف باستخدام Nvidia Nemotron (مجاناً) ---")

    openrouter_key = userdata.get('OpenRouter')


    llm = ChatOpenAI(
        model="stepfun/step-3.5-flash:free",
        openai_api_key=openrouter_key,
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0
    )


    my_profile = """
    الاسم: أريج الزايدي.
    المؤهل: خريجة تقنية معلومات من جامعة الطائف.
    الشهادات الاحترافية:
    - AWS Certified DevOps Engineer - Professional (تدريب 100 ساعة).
    - AWS Certified Cloud Practitioner.
    - eJPT (Junior Penetration Tester).
    - Security+.
    - COBIT 5 (GRC & Governance).
    الخبرات: التدريب في معسكر سدايا (SDA)، خبرة في سياسات NCA، وبرمجة Python و LangChain.
    الموقع المفضل: جدة أو مكة.
    """

    results = []


    for job in state['job_details']:
        prompt = f"""

        {my_profile}


        {job['content']}

        أعطني درجة مطابقة من 10 وملخصاً لنقاط القوة والنقص.
        """

        response = llm.invoke(prompt)

        results.append(f"الرابط: {job['url']}\nالتقييم:\n{response.content}\n")


    return {"final_report": "\n---\n".join(results)}

In [145]:
from langgraph.graph import StateGraph, END


workflow = StateGraph(AgentState)


workflow.add_node("extractor", extract_barcodes_node)
workflow.add_node("scraper", scrape_jobs_node)
workflow.add_node("matcher", match_evaluator_node)


workflow.set_entry_point("extractor")
workflow.add_edge("extractor", "scraper")
workflow.add_edge("scraper", "matcher")
workflow.add_edge("matcher", END)


app = workflow.compile()


inputs = {"folder_path": "/content/drive/MyDrive/barcodes"}


try:
    final_output = app.invoke(inputs)
    print("\n" + "="*30)
    print("--- التقرير النهائي ---")
    print("="*30)
    print(final_output.get('final_report', "لم يتم توليد تقرير."))
except Exception as e:
    print(f"حدث خطأ أثناء التشغيل: {e}")

--- Exreracting links from barcodes---
--- جاري تحليل 33 روابط ---
فشل الوصول للرابط: https://sysdown.sawaeed.sa/companySurvey/contact/home بسبب: HTTPSConnectionPool(host='sysdown.sawaeed.sa', port=443): Max retries exceeded with url: /companySurvey/contact/home (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7fce89659be0>, 'Connection to sysdown.sawaeed.sa timed out. (connect timeout=10)'))
فشل الوصول للرابط: ksahr@carrier.com بسبب: Invalid URL 'ksahr@carrier.com': No scheme supplied. Perhaps you meant https://ksahr@carrier.com?
فشل الوصول للرابط: https://mim.gov.sa/youth-growth-program/ بسبب: HTTPSConnectionPool(host='mim.gov.sa', port=443): Max retries exceeded with url: /youth-growth-program/ (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fce896a93a0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))
فشل الوصول للرابط: https://career.ncms.sa/ بسبب: HTTPSConnectionPool(host='career.ncms.sa', p